In [1]:
!pip install ultralytics torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
!nvidia-smi

True
Tesla T4
Sun May 10 13:31:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [3]:
import os
import shutil
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path
from ultralytics import YOLO

def get_cluttered_images(xml_path):
    """
    Parses the annotations.xml to find images tagged as 'cluttered'.
    Returns a set of image filenames.
    """
    if not xml_path.exists():
        print(f"Warning: {xml_path} not found. No images will be marked as cluttered.")
        return set()
    tree = ET.parse(xml_path)
    root = tree.getroot()
    cluttered_images = set()

    for image in root.findall('image'):
        image_name = image.get('name')
        for tag in image.findall('tag'):
            if tag.get('label') == 'cluttered':
                cluttered_images.add(image_name)
                break
    return cluttered_images

def preprocess_dataset(input_dir, output_dir, target_class_name="graffiti", dry_run=False):
    """
    Preprocesses the dataset:
    - Merges all labels into one (target_class_name).
    - Filters out images without bounding boxes.
    - Uses annotations.xml to separate 'cluttered' images for evaluation.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    xml_path = input_path / "annotations.xml"

    # Train split (clean images)
    train_img_out = output_path / "images" / "train"
    train_lb_out = output_path / "labels" / "train"
    # Val split (cluttered images)
    val_img_out = output_path / "images" / "val"
    val_lb_out = output_path / "labels" / "val"

    if not dry_run:
        for d in [train_img_out, train_lb_out, val_img_out, val_lb_out]:
            d.mkdir(parents=True, exist_ok=True)

    cluttered_images = get_cluttered_images(xml_path)
    if dry_run:
        print(f"[Dry Run] Found {len(cluttered_images)} cluttered images in XML.")
    else:
        print(f"Found {len(cluttered_images)} cluttered images in XML.")

    label_files = list((input_path / "labels" / "train").glob("*.txt"))
    train_count = 0
    val_count = 0
    skipped_empty = 0
    skipped_no_img = 0

    for lb_file in label_files:
        img_name_stem = lb_file.stem

        # Find corresponding image
        img_file = None
        img_full_name = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            candidate = input_path / "images" / "train" / f"{img_name_stem}{ext}"
            if candidate.exists():
                img_file = candidate
                img_full_name = f"{img_name_stem}{ext}"
                break

        if not img_file:
            skipped_no_img += 1
            continue

        with open(lb_file, 'r') as f:
            lines = f.readlines()

        if not lines:
            skipped_empty += 1
            continue

        new_labels = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            new_labels.append(f"0 {' '.join(parts[1:])}")

        # Decide split
        if img_full_name in cluttered_images:
            img_dest = val_img_out
            lb_dest = val_lb_out
            val_count += 1
        else:
            img_dest = train_img_out
            lb_dest = train_lb_out
            train_count += 1

        if not dry_run:
            shutil.copy(img_file, img_dest / img_file.name)
            with open(lb_dest / f"{img_name_stem}.txt", 'w') as f:
                f.write('\n'.join(new_labels) + '\n')

    print(f"Preprocessing results {'(Dry Run)' if dry_run else ''}:")
    print(f"- Training images (clean): {train_count}")
    print(f"- Validation images (cluttered): {val_count}")
    print(f"- Skipped empty: {skipped_empty}")
    print(f"- Skipped (no image found): {skipped_no_img}")

    if dry_run:
        return None

    # Create new data.yaml
    new_data_config = {
        'path': str(output_path.absolute()),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: target_class_name}
    }

    with open(output_path / "data.yaml", 'w') as f:
        yaml.dump(new_data_config, f)

    return output_path / "data.yaml"

def train_yolo(data_yaml, model_name="yolov8n.pt", epochs=50, imgsz=640, batch=16, device=None, **kwargs):
    if data_yaml is None:
        print("[Dry Run] Skipping training.")
        return None
    model = YOLO(model_name)
    results = model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        plots=True,
        single_cls=True,
        **kwargs
    )
    return results


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
DATASET_PATH = "/kaggle/input/datasets/luengor/graffiti-annotated-2"
SAVE_DIR = "/kaggle/working/yolo"
os.makedirs(SAVE_DIR, exist_ok=True)

IMAGE_SIZE = 1280
BATCH_SIZE = 8

AUGMENT = {
    # Color
    'hsv_h': 0.015,
    'hsv_s': 0.5,
    'hsv_v': 0.2,
    'bgr': 0,

    # Geometric
    'degrees': 15.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0001,
    'flipud': 0.0,
    'fliplr': 0.0,
    'mosaic': 1.0,
    'mixup': 0.0,
}

EPOCHS = 100
MODEL_NAME = "yolo11m.pt"

TRAINING = {
    'lr0': 0.001,
    'lrf': 0.01,

    'warmup_epochs': 5,
    'close_mosaic': 20,
    'patience': 20,
    'optimizer': 'AdamW',
}

LOSSES = {
    'box': 8,
    'cls': 0.4,
    'dfl': 1.0,
}

train_yolo(DATASET_PATH, model_name=MODEL_NAME, epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device = [-1, -1], **AUGMENT, **TRAINING, **LOSSES)

Searching for 2 idle GPUs with free memory >= 20.0% and free utilization >= 0.0%...
Selected idle CUDA devices [1, 0]
Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:1 (Tesla T4, 14913MiB)
                                                       CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0, box=8, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.4, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/luengor/graffiti-annotated-2, degrees=15.0, deterministic=True, device=1,0, dfl=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.2, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01,

In [6]:
import shutil

def zip_run_folder(run_id: int) -> str:
    filename = f"/kaggle/working/runs/detect/train"
    if run_id != 0:
        filename += f"-{run_id}"
    print(filename)
    shutil.make_archive(filename, 'zip', filename)
    return f"{filename}.zip"


In [7]:
zip_run_folder(10)

/kaggle/working/runs/detect/train-10


'/kaggle/working/runs/detect/train-10.zip'